In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio

# ============================================================
# 1) Global pilot field on the unwrapped torus (thetaA, thetaB)
# ============================================================
# Psi(d) = cos(d) + i*eps*sin(d)
# so |Psi|^2 = cos^2(d) + eps^2 sin^2(d)
# and phase phi(d) = arg(Psi) = atan2(eps sin d, cos d)
# "frequency pull" ~ d(phi)/d(d)
eps = 0.30

def psi_mag2(d):
    return (np.cos(d)**2 + (eps**2)*(np.sin(d)**2))

def dphi_dd(d):
    # derivative of atan2(eps sin d, cos d)
    denom = (np.cos(d)**2 + (eps**2)*(np.sin(d)**2))
    return eps / denom

# ============================================================
# 2) Settings (fixed for this demo animation)
# ============================================================
a = 0.0          # Alice analyzer / reference angle
b = np.pi/8      # Bob analyzer / reference angle

# ============================================================
# 3) Dynamics: two local rotors + two bistable pointers
# ============================================================
omega0 = 1.0     # base rotor angular velocity
k_pull = 0.55    # strength of nonlocal frequency pull from pilot field gradient

# pointer latch: double-well + drive from local rotor alignment to analyzer
eta_drive = 1.2
gamma = 1.0

def g_local(theta, setting):
    # local "alignment" signal (your analyzer projection analogue)
    return np.cos(theta - setting)

def step(state, dt):
    """
    state = [thetaA, thetaB, yA, yB]
    thetaA/thetaB are local rotor angles
    yA/yB are bistable pointers that drift to +/- basins
    """
    thetaA, thetaB, yA, yB = state

    # joint phase coordinate on the unwrapped torus (include settings)
    d = (thetaA - thetaB) - 2*(a - b)

    # "frequency pull" from pilot-phase gradient
    pull = dphi_dd(d)
    thetaA_dot = omega0 + k_pull*pull
    thetaB_dot = omega0 - k_pull*pull

    # bistable pointer latches: V(y)=(y^2-1)^2/4 -> dV/dy=y(y^2-1)
    yA_dot = -gamma*(yA*(yA**2 - 1)) + eta_drive*g_local(thetaA, a)
    yB_dot = -gamma*(yB*(yB**2 - 1)) + eta_drive*g_local(thetaB, b)

    thetaA = (thetaA + dt*thetaA_dot) % (2*np.pi)
    thetaB = (thetaB + dt*thetaB_dot) % (2*np.pi)
    yA = yA + dt*yA_dot
    yB = yB + dt*yB_dot

    return np.array([thetaA, thetaB, yA, yB], dtype=float), pull

# ============================================================
# 4) Balanced 3-phase rendering (visual only)
# ============================================================
def abc_waveforms(theta, t_fast, omega_fast=10.0):
    va = np.cos(omega_fast*t_fast + theta)
    vb = np.cos(omega_fast*t_fast + theta - 2*np.pi/3)
    vc = np.cos(omega_fast*t_fast + theta + 2*np.pi/3)
    return va, vb, vc

def alphabeta_from_abc(va, vb, vc):
    # simple Clarke for arrow visualization (scaling doesn't matter)
    v_alpha = (2/3)*(va - 0.5*vb - 0.5*vc)
    v_beta  = (2/3)*(np.sqrt(3)/2)*(vb - vc)
    return v_alpha, v_beta

# ============================================================
# 5) Precompute the pilot heatmap (torus unwrapped)
# ============================================================
n_grid = 140
th = np.linspace(0, 2*np.pi, n_grid, endpoint=False)
THA, THB = np.meshgrid(th, th, indexing="xy")
MAG2 = psi_mag2((THA - THB) - 2*(a - b))

# ============================================================
# 6) Simulate a trajectory (one hidden microstate)
# ============================================================
T = 4.0
dt = 0.02
n_steps = int(T/dt)

traj = np.zeros((n_steps+1, 4), dtype=float)
pulls = np.zeros(n_steps+1, dtype=float)
ts = np.linspace(0, T, n_steps+1)

# Initial conditions (the “hidden” microstate)
traj[0] = np.array([0.9, 4.8, 0.05, -0.02], dtype=float)
for k in range(n_steps):
    traj[k+1], pulls[k+1] = step(traj[k], dt)

# ============================================================
# 7) Render frames and write GIF
# ============================================================
t_fast = np.linspace(0, 2*np.pi, 240)

# Choose how many frames to export (keep small for fast GIF)
n_frames = 24
frame_ids = np.linspace(0, n_steps, n_frames).astype(int)

frames = []
for k in frame_ids:
    thetaA, thetaB, yA, yB = traj[k]
    tnow = ts[k]
    pull = pulls[k]

    # Waveforms
    vaA, vbA, vcA = abc_waveforms(thetaA, t_fast)
    vaB, vbB, vcB = abc_waveforms(thetaB, t_fast)

    # αβ arrows from instantaneous sample
    aA, bA = alphabeta_from_abc(vaA[0], vbA[0], vcA[0])
    aB, bB = alphabeta_from_abc(vaB[0], vbB[0], vcB[0])

    # Make a 2x2 panel figure
    fig = plt.figure(figsize=(9.5, 5.3))
    gs = fig.add_gridspec(2, 2)

    axA = fig.add_subplot(gs[0, 0])
    axB = fig.add_subplot(gs[0, 1])
    axH = fig.add_subplot(gs[1, 0])
    axP = fig.add_subplot(gs[1, 1])

    # --- Alice panel ---
    axA.plot(t_fast, vaA, lw=1.0, label="Va")
    axA.plot(t_fast, vbA, lw=1.0, label="Vb")
    axA.plot(t_fast, vcA, lw=1.0, label="Vc")
    axA.set_xlim(0, 2*np.pi); axA.set_ylim(-1.25, 1.25)
    axA.grid(True, alpha=0.25)
    axA.set_title("Alice abc")
    axA.set_xlabel("fast time")
    axA.legend(frameon=False, fontsize=8, loc="upper right")

    # arrow in corner
    x0, y0, sc = 5.15, 0.9, 0.55
    axA.annotate("", xy=(x0 + sc*aA, y0 + sc*bA), xytext=(x0, y0),
                 arrowprops=dict(arrowstyle="->", lw=2))

    # --- Bob panel ---
    axB.plot(t_fast, vaB, lw=1.0, label="Va")
    axB.plot(t_fast, vbB, lw=1.0, label="Vb")
    axB.plot(t_fast, vcB, lw=1.0, label="Vc")
    axB.set_xlim(0, 2*np.pi); axB.set_ylim(-1.25, 1.25)
    axB.grid(True, alpha=0.25)
    axB.set_title("Bob abc")
    axB.set_xlabel("fast time")
    axB.legend(frameon=False, fontsize=8, loc="upper right")

    axB.annotate("", xy=(x0 + sc*aB, y0 + sc*bB), xytext=(x0, y0),
                 arrowprops=dict(arrowstyle="->", lw=2))

    # --- Torus heatmap panel ---
    axH.imshow(MAG2, origin="lower", extent=[0, 2*np.pi, 0, 2*np.pi], aspect="auto")
    axH.plot([thetaA], [thetaB], marker="o", markersize=6)
    axH.set_xlabel("θA"); axH.set_ylabel("θB")
    axH.set_title("|Ψ(θA, θB)|²  (torus unwrapped)")

    # --- Pointer panel ---
    axP.plot(ts[:k+1], traj[:k+1, 2], lw=1.2, label="yA")
    axP.plot(ts[:k+1], traj[:k+1, 3], lw=1.2, label="yB")
    axP.plot([tnow], [yA], marker="o", markersize=5)
    axP.plot([tnow], [yB], marker="o", markersize=5)
    axP.axhline(0, ls="--", lw=1)
    axP.grid(True, alpha=0.25)
    axP.set_xlim(0, T); axP.set_ylim(-2.0, 2.0)
    axP.set_title("Pointers (votes)")
    axP.set_xlabel("t")
    axP.legend(frameon=False, fontsize=8, loc="upper right")

    voteA = "+1" if yA >= 0 else "-1"
    voteB = "+1" if yB >= 0 else "-1"
    # Add extra top margin so the suptitle does not overlap subplot titles
    fig.subplots_adjust(top=0.86, hspace=0.55, wspace=0.28)
    fig.suptitle(
        f"Global field pulls rotor rates: pull={pull:.2f}   t={tnow:.2f}s   votes≈({voteA},{voteB})",
        y=0.98, fontsize=11
    )

    # Convert figure to an RGB array
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    # Convert RGBA to RGB by dropping the alpha channel
    img = img[:, :, :3]
    frames.append(img)

# Write GIF
out_path = "bohmian_3phase_torus.gif"
imageio.mimsave(out_path, frames, duration=0.12)
print("Wrote:", out_path)
